In [ ]:
!pip install google-search-results pandas python-dotenv requests trafilatura youtube_transcript_api

^C


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached charset_normalizer-3.5.1-cp311-cp311-win_amd64.whl.metadata (46 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached babel-2.18.0-p


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


- **GoogleSearch (SerpAPI)** → Used to search Google through the SerpAPI.
- **YouTubeTranscriptApi** → Used to extract YouTube video transcripts.
- **ThreadPoolExecutor** → Allows multiple tasks to run concurrently, making scraping faster.
- **trafilatura** → Extracts the main text/content from web pages.
- **pandas** → Used for storing and processing data in DataFrames.
- **load_dotenv** → Loads environment variables from a `.env` file.
- **os** → Accesses environment variables and operating-system functionality.
- **re** → Used for regular expressions, usually for cleaning or extracting text.
- **time** → Used for delays or timing operations.

In [1]:
from serpapi import GoogleSearch
from youtube_transcript_api import YouTubeTranscriptApi
from concurrent.futures import ThreadPoolExecutor, as_completed
import trafilatura
import pandas as pd
from dotenv import load_dotenv
import os
import re
import time

load_dotenv()
SERP_API_KEY = os.getenv("SERP_API_KEY")

# Max number of articles to scrape
MAX_NUM_RESULTS = 10

# --- Sanity check: make sure the key actually loaded ---
if not SERP_API_KEY:
    raise ValueError(
        "SERPAPI_KEY not found. Check that your .env file exists in the "
        "current working directory and contains a line like: SERPAPI_KEY=your_key_here"
    )
else:
    print(f"✅ API key loaded")

✅ API key loaded


## `collect_search_results()` — Easy Explanation

This function **searches Google for multiple queries** using SerpAPI and collects both **articles and YouTube videos**.

### Step-by-step:

- **Takes queries as input**
  - `queries` → A list of search queries.
  - `num_results` → Maximum number of results to collect for each query.

- **Creates empty lists**
  - `article_rows` → Stores article information.
  - `video_rows` → Stores YouTube video information.
  - `raw_results` → Stores the complete SerpAPI responses for inspection.

- **Loops through each query**
  - Searches Google using SerpAPI.
  - Prints the query currently being searched.

- **Creates search parameters**
  - Specifies Google as the search engine.
  - Adds the search query.
  - Sets language and location.
  - Uses the SerpAPI key.

- **Gets Google search results**
  - `GoogleSearch(params)` performs the search.
  - `get_dict()` converts the response into a Python dictionary.

- **Collects organic results**
  - Gets normal Google search results using `organic_results`.
  - For each article, it stores:
    - Title
    - Snippet
    - Source
    - Date
    - URL
    - Type (`article`)

- **Collects YouTube videos**
  - Gets videos from `inline_videos`.
  - Stores:
    - Video title
    - Channel name
    - URL
    - Type (`video`)

- **Combines everything**
  - Articles and videos are combined into one list.
  - The list is converted into a **Pandas DataFrame**.

- **Handles empty results**
  - If nothing was found, it prints a warning and returns an empty DataFrame.

- **Removes duplicates**
  - `drop_duplicates(subset="url")` removes duplicate results based on their URL.

- **Returns two things**
  - `df` → DataFrame containing all collected articles and videos.
  - `raw_results` → Complete raw SerpAPI responses for inspection.

### In simple words:

**Queries → Google Search → Collect Articles + YouTube Videos → Store in DataFrame → Remove Duplicates → Return Results**

In [2]:
def collect_search_results(queries: list[str],
                            num_results: int = MAX_NUM_RESULTS) -> tuple[pd.DataFrame, list[dict]]:
    """
    Search Google for each query and collect:
    - organic_results  → articles (title, snippet, url, source, date)
    - inline_videos    → YouTube videos (title, channel, url)

    Returns:
        articles_df  : DataFrame of all article metadata
        raw_results  : list of raw SerpAPI result dicts (just for inspection)
    """
    article_rows = []
    video_rows   = []
    raw_results  = []

    for query in queries:
        print(f"\n🔍 Searching: '{query}'")

        params = {
            "engine":        "google",
            "q":              query,
            "google_domain": "google.com",
            "hl":            "en",
            "gl":            "us",
            "api_key":       SERP_API_KEY
        }

        search  = GoogleSearch(params)
        results = search.get_dict()
        raw_results.append({"query": query, "results": results})

        # — Organic articles —
        organic = results.get("organic_results", [])
        if not organic:
            print(f"  ⚠ No organic_results for: {query}")
        else:
            for article in organic[:num_results]:
                article_rows.append({
                    "query":   query,
                    "title":   article.get("title"),
                    "snippet": article.get("snippet"),
                    "source":  article.get("source"),
                    "date":    article.get("date"),
                    "url":     article.get("link"),
                    "type":    "article"
                })
            print(f"  ✅ {len(organic[:num_results])} articles collected")

        # — Inline videos —
        videos = results.get("inline_videos", [])
        if videos:
            for video in videos:
                video_rows.append({
                    "query":   query,
                    "title":   video.get("title"),
                    "snippet": None,
                    "source":  video.get("channel"),
                    "date":    None,
                    "url":     video.get("link"),
                    "type":    "video"
                })
            print(f"  ✅ {len(videos)} videos collected")

    # Combine into single DataFrame
    all_rows = article_rows + video_rows
    df = pd.DataFrame(all_rows)

    if df.empty:
        print("\n⚠ No results collected across all queries.")
        return df, raw_results

    # Remove duplicates by URL
    df = df.drop_duplicates(subset="url").reset_index(drop=True)

    return df, raw_results

## Collect URLs from Search Queries

- `collect_search_results()` searches Google for the given queries.
- Here, we are searching for:
  - **AI intellectual property**
  - **copyright Generative AI**
- The search results are stored in `df`.
- The raw SerpAPI results are stored in `raw`.
- `df.head()` displays the **first 5 results** from `df`.

### In simple words:

**Search queries → Google search → Results stored in `df` → Show first 5 results**

In [3]:
# — Collect urls from search query —
df, raw = collect_search_results(["AI intellectual property",
                                   "copyright Generative AI"])
df.head()


🔍 Searching: 'AI intellectual property'
  ✅ 9 articles collected

🔍 Searching: 'copyright Generative AI'
  ✅ 9 articles collected


,query,title,snippet,source,date,url,type
0,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI is rapidly transforming the creative and in...,WIPO - World Intellectual Property Organization,NaN,https://www.wipo.int/en/web/frontier-technolog...,article
2,AI intellectual property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article
3,AI intellectual property,What's yours isn't mine: AI and intellectual p...,If AI uses their intellectual property without...,JHU Carey Business School,"Jun 14, 2024",https://carey.jhu.edu/news/whats-yours-isnt-mi...,article
4,AI intellectual property,Generative Artificial Intelligence and Copyrig...,The AI Guidance states that authors may claim ...,Congress.gov,"Jul 18, 2025",https://www.congress.gov/crs-product/LSB10922,article


In [4]:
raw

[{'query': 'AI intellectual property',
  'results': {'search_metadata': {'id': '6a86ca3e2b15b909d4544aaa',
    'status': 'Success',
    'json_endpoint': 'https://serpapi.com/searches/Ceb6S0SdsbU5rTMD7LozVA/6a86ca3e2b15b909d4544aaa.json',
    'markdown_endpoint': 'https://serpapi.com/searches/Ceb6S0SdsbU5rTMD7LozVA/6a86ca3e2b15b909d4544aaa.md',
    'pixel_position_endpoint': 'https://serpapi.com/searches/Ceb6S0SdsbU5rTMD7LozVA/6a86ca3e2b15b909d4544aaa.json_with_pixel_position',
    'created_at': '2026-08-20 09:34:54 UTC',
    'processed_at': '2026-08-20 09:34:54 UTC',
    'google_url': 'https://www.google.com/search?q=AI+intellectual+property&oq=AI+intellectual+property&hl=en&gl=us&sourceid=chrome&ie=UTF-8',
    'raw_html_file': 'https://serpapi.com/searches/Ceb6S0SdsbU5rTMD7LozVA/6a86ca3e2b15b909d4544aaa.html',
    'total_time_taken': 0.86},
   'search_parameters': {'engine': 'google',
    'q': 'AI intellectual property',
    'google_domain': 'google.com',
    'hl': 'en',
    'gl': 'us



**URLs collected earlier → Articles are scraped + Videos are transcribed → Full content is added to the DataFrame**

So, this cell is the **enrichment step**: it turns the basic search results into results containing the **actual content**.

In [5]:
# Define scrapers
def _scrape_url(url: str) -> dict:
    """Scrape clean article text from a URL using Trafilatura."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"url": url, "full_text": None,
                     "status": "failed_download"}

        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=True,
            no_fallback=False,
            favor_precision=False,
            deduplicate=True
        )
        return {
            "url":       url,
            "full_text": text.strip() if text else None,
            "status":    "success" if text else "failed_extraction"
        }
    except Exception as e:
        return {"url": url, "full_text": None,
                 "status": f"error: {str(e)}"}


def _get_youtube_id(url: str) -> str | None:
    """Extract YouTube video ID from URL."""
    patterns = [
        r"youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})",
        r"youtube\.com/shorts/([a-zA-Z0-9_-]{11})",
        r"youtu\.be/([a-zA-Z0-9_-]{11})"
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def _get_transcript(video_id: str) -> str | None:
    """Fetch YouTube transcript from video ID."""
    try:
        transcript = YouTubeTranscriptApi.get_transcript(
            video_id,
            languages=["en", "en-US", "en-GB"]
        )
        return " ".join([t["text"] for t in transcript])
    except Exception:
        return None


# ---------------------------
# ENRICHMENT
# ---------------------------

def enrich_search_results(df: pd.DataFrame,
                           max_workers: int = 5,
                           delay: float = 1.0) -> pd.DataFrame:
    """
    Enrich a DataFrame from collect_search_results() with full text content.
    - Rows with type='article' → scraped via Trafilatura
    - Rows with type='video'   → transcript via YouTube Transcript API

    Args:
        df:          DataFrame returned by collect_articles()
        max_workers: concurrent threads for article scraping
        delay:       seconds between requests per thread

    Returns:
        Enriched DataFrame with added columns:
        - full_text  : scraped article text OR video transcript
        - video_id   : YouTube video ID (videos only)
        - status     : success / failed_download / no_transcript / etc.
    """
    if df.empty:
        print("⚠ Empty DataFrame — nothing to enrich.")
        return df

    df = df.copy()
    df["full_text"] = None
    df["video_id"]  = None
    df["status"]    = "pending"

    # — Split by type —
    article_mask = df["type"] == "article"
    video_mask   = df["type"] == "video"

    article_df = df[article_mask].copy()
    video_df   = df[video_mask].copy()

    # — Scrape articles —
    if not article_df.empty:
        print(f"\n📄 Scraping {len(article_df)} articles...")

        url_to_result = {}

        # ⚠️ RECONSTRUCTED — not visible in your screenshots.
        # Verify this block against your actual notebook.
        unique_urls = article_df["url"].dropna().unique().tolist()

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(_scrape_url, url): url for url in unique_urls}

            for future in as_completed(futures):
                url = futures[future]
                result = future.result()
                url_to_result[url] = result
                time.sleep(delay)

        for idx, row in article_df.iterrows():
            result = url_to_result.get(row["url"])
            if result:
                df.at[idx, "full_text"] = result["full_text"]
                df.at[idx, "status"]    = result["status"]
            else:
                df.at[idx, "status"] = "failed_download"

            icon  = "✅" if df.at[idx, "status"] == "success" else "❌"
            title = (row["title"] or "")[:50]
            print(f"  {icon} {title}... [{df.at[idx, 'status']}]")

        success = sum(
            1 for idx in article_df.index
            if df.at[idx, "status"] == "success"
        )
        print(f"\n📄 Articles: {success}/{len(article_df)} scraped successfully")

    # — Scrape video transcripts —
    if not video_df.empty:
        print(f"\n🎥 Fetching {len(video_df)} video transcripts...")

        for idx, row in video_df.iterrows():
            vid_id = _get_youtube_id(row["url"])

            transcript = None
            if vid_id:
                transcript = _get_transcript(vid_id)

            status = "success" if transcript else "no_transcript"
            icon   = "✅" if transcript else "❌"

            df.at[idx, "video_id"]  = vid_id
            df.at[idx, "full_text"] = transcript
            df.at[idx, "status"]    = status

            title = (row["title"] or "")[:50]
            print(f"  {icon} {title}... [{status}]")

        success = sum(
            1 for idx in video_df.index
            if df.at[idx, "status"] == "success"
        )
        print(f"\n🎥 Videos: {success}/{len(video_df)} "
              f"transcripts fetched")

    return df

**Search results → Get full content → Remove failed results → View first 5 successful results**

So, `df` contains **all results**, while `df_clean` contains **only successfully collected content**.

In [6]:
# —— Enrich data with full texts from articles and videos ——
df = enrich_search_results(df)

# —— Filter to only successfully scraped content ——
df_clean = df[df["status"] == "success"].reset_index(drop=True)

# —— Inspect results ——
df_clean.head()


📄 Scraping 16 articles...
  ✅ AI, Copyright, and the Law: The Ongoing Battle Ove... [success]
  ✅ Artificial Intelligence and Intellectual Property... [success]
  ✅ Generative AI: Navigating intellectual property... [success]
  ❌ What's yours isn't mine: AI and intellectual prope... [failed_download]
  ❌ Generative Artificial Intelligence and Copyright L... [failed_download]
  ❌ IP in the Age of AI: What Today's Cases Teach Us A... [failed_download]
  ✅ Intellectual Property Rights and AI-Generated Cont... [success]
  ✅ Artificial intelligence and intellectual property:... [success]
  ✅ Generative Artificial Intelligence: Intellectual P... [success]
  ✅ Copyright and Artificial Intelligence | U.S. Copyr... [success]
  ✅ Copyright and Generative AI... [success]
  ✅ AI Tools and Resources: Copyright and Generative A... [success]
  ✅ The Law and Economics of Generative AI and Copyrig... [success]
  ✅ Copyright and Intellectual Property - Generative A... [success]
  ✅ Generative AI Is a C

,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,None,success
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI is rapidly transforming the creative and in...,WIPO - World Intellectual Property Organization,NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,None,success
2,AI intellectual property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,None,success
3,AI intellectual property,Intellectual Property Rights and AI-Generated ...,The rise of generative AI is stress-testing ma...,"Medium · Adnan Masood, PhD.",NaN,https://medium.com/@adnanmasood/intellectual-p...,article,Member-only story\nIntellectual Property Right...,None,success
4,AI intellectual property,Artificial intelligence and intellectual prope...,An AI can generate a logo or visual close to a...,Cabinet Dreyfus,"Oct 6, 2025",https://www.dreyfus.fr/en/2025/10/06/artificia...,article,"A technological revolution, a legal vacuum\nAr...",None,success


In [7]:
df_clean

,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,None,success
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI is rapidly transforming the creative and in...,WIPO - World Intellectual Property Organization,NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,None,success
2,AI intellectual property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,None,success
3,AI intellectual property,Intellectual Property Rights and AI-Generated ...,The rise of generative AI is stress-testing ma...,"Medium · Adnan Masood, PhD.",NaN,https://medium.com/@adnanmasood/intellectual-p...,article,Member-only story\nIntellectual Property Right...,None,success
4,AI intellectual property,Artificial intelligence and intellectual prope...,An AI can generate a logo or visual close to a...,Cabinet Dreyfus,"Oct 6, 2025",https://www.dreyfus.fr/en/2025/10/06/artificia...,article,"A technological revolution, a legal vacuum\nAr...",None,success
5,AI intellectual property,Generative Artificial Intelligence: Intellectu...,"As of right now, copyright can only be granted...",United States Military Academy West Point,NaN,https://guides.library.westpoint.edu/c.php?g=1...,article,Jefferson Hall Library\n\t\t\tLoading…\n\t\t ...,None,success
6,copyright Generative AI,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,None,success
7,copyright Generative AI,Copyright and Generative AI,The legal framework governing copyright faces ...,The Regulatory Review,"Jun 7, 2025",https://www.theregreview.org/2025/06/07/semina...,article,Scholars discuss emerging problems at the inte...,None,success
8,copyright Generative AI,AI Tools and Resources: Copyright and Generati...,Generative AI tools can be used to infringe on...,University of South Florida,"Jul 17, 2026",https://guides.lib.usf.edu/AI/copyright,article,Copyright is a form of intellectual property p...,None,success
9,copyright Generative AI,The Law and Economics of Generative AI and Cop...,Generative AI challenges copyright law at both...,National Institutes of Health (NIH) | (.gov),NaN,https://pmc.ncbi.nlm.nih.gov/articles/PMC12658...,article,Checking your browser before accessing pmc.ncb...,None,success


In [8]:
df_clean.to_csv("ai_copyright_dataset.csv", index=False)